In [5]:
print(123)

123


In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [8]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [9]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test that the local server is running, use:
   ```bash
   curl http://localhost:11434
   ```

If you want to use it from Python, install the client with:
```bash
pip install ollama
```


In [10]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any information in the provided context about running **Olama/Olama locally**.

If you meant **MCP Inspector**, the command in the context is:

```bash
npx @modelcontextprotocol/inspector
```




In [11]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Absolutely — if the course is still open, you can usually join it.\n\nA quick reply you could send is:\n> Hi, I just discovered the course and I’m very interested. Is it still possible to join?\n\nIf you want, I can also help you make this message:\n- more polite\n- more casual\n- or tailored for email / chat / a teacher or organizer'

In [12]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [13]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [14]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [15]:
call = response.output[0]

In [16]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll can I join discovered the course"}', call_id='call_MiDh7Q0gBy2WHdj1vUOEEjzg', name='search', type='function_call', id='fc_01e4de1b1089c35f006a2e8f153870819186a3aa18ce0517e8', namespace=None, status='completed')

In [17]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered late enroll can I join discovered the course'}

In [18]:
results = search(**args)

In [19]:
result_json = json.dumps(results, indent=2)

In [20]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n\\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n   

In [21]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [22]:
messages.append(call)

In [23]:
messages.append(function_call_output)

In [24]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enroll can I join discovered the course"}', call_id='call_MiDh7Q0gBy2WHdj1vUOEEjzg', name='search', type='function_call', id='fc_01e4de1b1089c35f006a2e8f153870819186a3aa18ce0517e8', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_MiDh7Q0gBy2WHdj1vUOEEjzg',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a cert

In [25]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [26]:
print(response.output)

[ResponseOutputMessage(id='msg_01e4de1b1089c35f006a2e8f16dc908191be5d73e3278c94da', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, you need to submit your project while submissions are still being accepted.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


In [27]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [28]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [29]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [30]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you need to submit your project while submissions are still being accepted.


In [31]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseOutputMessage(id='msg_01e4de1b1089c35f006a2e8f16dc908191be5d73e3278c94da', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf you want a certificate, you need to submit your project while submissions are still being accepted.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [32]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...


function_call: search {"query":"join course discovered can I join enrolled late registration FAQ"}
function_call: search {"query":"course discovered join enrollment registration access FAQ"}
function_call: search {"query":"late join course can I still enroll FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join.

If you want a certificate, you’ll need to submit your project while submissions are still open. If you’re just starting now, you can also learn at your own pace, but certificates are only available for the live cohort.

If you want, I can also help you with:
- whether you need to register,
- how certificates work,
- or how to catch up quickly.


In [33]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [34]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course enrollment discovered course can I join"}
function_call: search {"query":"course FAQ join register enrollment late join discovered course"}
function_call: search {"query":"new student join course after start enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you’re just starting late, you can still learn from the materials; the certificate is tied to the live cohort timing.

If you want, I can also help with how registration, homework submission, or certificates work.


In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

result = agent_loop_langchain(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment registration late join"}


TypeError: 'StructuredTool' object is not callable

In [36]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [37]:

from langchain.tools import tool



In [38]:
@tool
def search(query):
    """Search the FAQ database for entries matching the given query."""
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [39]:
from langchain_openai import ChatOpenAI


In [40]:
llm = ChatOpenAI(
    model_name='gpt-5.4-mini',
    temperature=0.0,
)

In [41]:
llm_with_tools = llm.bind_tools([search])

In [42]:
response = llm_with_tools.invoke(
    "I just discovered the course. Can I join it?"
)


In [49]:
print(response.tool_calls)

[{'name': 'search', 'args': {'query': 'join course discovered late enrollment can I join course FAQ'}, 'id': 'call_9yKLloVqm7j2ceSsZeakUFeU', 'type': 'tool_call'}]


In [70]:
from langchain_core.messages import ToolMessage

def make_call_langchain(call):
    args = call['args']

    if call['name'] == 'search':
        result = search.invoke(args)

    result_json = json.dumps(result, indent=2)

    return ToolMessage(
          content= str(result_json),
          tool_call_id=call['id']
    )

In [78]:
from langchain_core.messages import HumanMessage

def agent_loop_langchain(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        HumanMessage(content=instructions),
        HumanMessage(content=question)
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = llm_with_tools.invoke(
                messages
            )


        messages.append(response)

        if not response.tool_calls:
           print ('No tool calls, breaking the loop.')
           print('Final answer:', response.content)
           return response.content

        for item in response.tool_calls:
            if item['type'] == 'tool_call':
                print('tool_call:', item['name'], item['args'])
                call_output = make_call_langchain(item)
                messages.append(call_output)
                has_function_calls = True


        it = it + 1
        if has_function_calls == False:
            break
    
    return response.content

In [79]:
result = agent_loop_langchain(instructions, question)


iteration #1...
tool_call: search {'query': 'course join discovered can I join enrollment late registration join course'}
iteration #2...
tool_call: search {'query': 'join course still join discovered course can I still join certificate submit project while accepting submissions self-paced live cohort'}
iteration #3...
No tool calls, breaking the loop.
Final answer: Yes — you can still join the course. If you want a certificate, make sure you submit your project while submissions are still being accepted.

If you’d like, I can also help with questions about certificates, homework, or project submission. Are there other areas that you want to explore?


In [81]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=[search]
)

In [ ]:
messages = [
        HumanMessage(content=instructions),
        HumanMessage(content=question)
    ]

response = agent.invoke(
    {
        "messages": messages
    }
)

print(response['AIMessage'].content)

KeyError: 'AImessage'